In [ ]:
import pkgutil, importlib
import cuda.pathfinder as _pkg_under_test
failed = []
for module_info in pkgutil.walk_packages(_pkg_under_test.__path__, prefix=f"{_pkg_under_test.__name__}."):
    try: importlib.import_module(module_info.name)
    except Exception as e: failed.append((module_info.name, type(e).__name__, str(e)))
if failed:
    for name, etype, msg in failed: print(f"FAIL {name}: {etype}: {msg}")
    raise AssertionError(f"{len(failed)} submodule(s) of {_pkg_under_test.__name__} failed to import")
print(f"OK — all submodules of {_pkg_under_test.__name__} imported cleanly")

In [ ]:
# Example 1: version + the five SUPPORTED_* capability catalogs.
import cuda.pathfinder as cpf

print("cuda.pathfinder version:", cpf.__version__)
assert isinstance(cpf.__version__, str) and cpf.__version__[0].isdigit()

catalogs = {
    "SUPPORTED_NVIDIA_LIBNAMES":  cpf.SUPPORTED_NVIDIA_LIBNAMES,
    "SUPPORTED_STATIC_LIBS":      cpf.SUPPORTED_STATIC_LIBS,
    "SUPPORTED_BITCODE_LIBS":     cpf.SUPPORTED_BITCODE_LIBS,
    "SUPPORTED_HEADERS_CTK":      cpf.SUPPORTED_HEADERS_CTK,
    "SUPPORTED_BINARY_UTILITIES": cpf.SUPPORTED_BINARY_UTILITIES,
}
for name, cat in catalogs.items():
    assert len(cat) > 0, f"{name} is empty"
    print(f"{name}: {len(cat)} entries e.g. {list(cat)[:4]}")

# Core CTK runtime libs must be advertised as locatable.
for essential in ("cudart", "nvrtc", "cublas"):
    assert essential in cpf.SUPPORTED_NVIDIA_LIBNAMES, essential
print("OK — version + capability catalogs present")

In [ ]:
# Example 2: locate CUDA Toolkit header directories (CPU-safe, no GPU needed).
import os
import cuda.pathfinder as cpf

cuda_root = cpf.get_cuda_path_or_home()
print("CUDA path/home:", cuda_root)

for comp in ("cudart", "cublas", "cufft"):
    hdr = cpf.find_nvidia_header_directory(comp)          # bare path str | None
    rec = cpf.locate_nvidia_header_directory(comp)        # rich record | None
    if hdr is None:
        print(f"SKIP {comp}: headers not installed")
        continue
    assert os.path.isdir(hdr), f"{comp} header dir does not exist: {hdr}"
    assert rec is not None and rec.abs_path == hdr
    print(f"{comp}: {hdr}  (found_via={rec.found_via})")
print("OK — header discovery resolved to real on-disk directories")

In [ ]:
# Example 3: locate CUDA binary utilities and static libraries on disk.
import os
import cuda.pathfinder as cpf

found_bins = 0
for util in ("nvcc", "cuobjdump", "nvdisasm"):
    path = cpf.find_nvidia_binary_utility(util)            # abs path | None
    if path is None:
        print(f"SKIP {util}: not installed")
        continue
    assert os.path.isfile(path) and os.access(path, os.X_OK), path
    found_bins += 1
    print(f"{util}: {path}")

sname = cpf.SUPPORTED_STATIC_LIBS[0]                       # 'cudadevrt'

cuda_root_variable = next((name for name in ("CUDA_HOME", "CUDA_PATH") if os.environ.get(name)), None)
assert cuda_root_variable is not None, (
    "neither CUDA_HOME nor CUDA_PATH is set, so cuda.pathfinder has no toolkit root to search and "
    "will report libcudadevrt.a missing however present it is. Set one of them to the CUDA Toolkit root."
)
print(f"CUDA root from {cuda_root_variable}={os.environ[cuda_root_variable]}")
spath = cpf.find_static_lib(sname)
srec = cpf.locate_static_lib(sname)
assert os.path.isfile(spath) and spath.endswith(".a"), spath
assert srec.abs_path == spath
print(f"static {sname}: {spath}")
print(f"OK — {found_bins} binary utilities + static lib located")

In [ ]:
# Example 4: RUNTIME INIT — dlopen real CUDA .so files into the process (GPU-less OK).
import ctypes, os
import cuda.pathfinder as cpf

loaded = {}
for lib in ("cudart", "nvrtc"):          # a genuine dlopen; fails if the .so is mislinked
    dl = cpf.load_nvidia_dynamic_lib(lib)
    assert os.path.isfile(dl.abs_path), dl.abs_path
    loaded[lib] = dl
    print(f"loaded {lib}: {dl.abs_path} (found_via={dl.found_via})")

# The loader is a process-wide cache: reloading returns the same handle.
assert cpf.load_nvidia_dynamic_lib("nvrtc")._handle_uint == loaded["nvrtc"]._handle_uint

# Prove the handle is a live, callable library: query the NVRTC version.
nvrtc = ctypes.CDLL(loaded["nvrtc"].abs_path)
major, minor = ctypes.c_int(), ctypes.c_int()
assert nvrtc.nvrtcVersion(ctypes.byref(major), ctypes.byref(minor)) == 0
print(f"NVRTC runtime version: {major.value}.{minor.value}")

# GPU-conditional: use loaded cudart to count visible devices. Skip only when there is no device
# (cudaErrorNoDevice) or no driver to reach one (cudaErrorInsufficientDriver); any other code fails.
cudaSuccess, cudaErrorNoDevice, cudaErrorInsufficientDriver = 0, 100, 35   # cudaError_t values
cudart, n = ctypes.CDLL(loaded["cudart"].abs_path), ctypes.c_int()
rc = cudart.cudaGetDeviceCount(ctypes.byref(n))
if rc in (cudaErrorNoDevice, cudaErrorInsufficientDriver) or (rc == cudaSuccess and n.value == 0):
    print(f"SKIP (no GPU): cudart loaded, cudaGetDeviceCount -> {rc}, {n.value} device(s)")
else:
    assert rc == cudaSuccess, f"cudaGetDeviceCount failed with cudaError_t {rc}"
    print(f"GPU present: {n.value} device(s)")

In [ ]:
# Example 5: error taxonomy — unsupported names vs supported-but-absent libs.
import cuda.pathfinder as cpf

try:                                   # unsupported libname
    cpf.load_nvidia_dynamic_lib("not_a_real_lib")
except cpf.DynamicLibUnknownError as e:
    print("unsupported dynamic lib ->", type(e).__name__)
else:
    raise AssertionError("expected DynamicLibUnknownError")

try:                                   # unsupported static-lib name -> ValueError
    cpf.find_static_lib("not_a_real_static")
except ValueError as e:
    print("unsupported static lib ->", type(e).__name__)
else:
    raise AssertionError("expected ValueError")

try:
    dl = cpf.load_nvidia_dynamic_lib("cudla")
    print("cudla present ->", dl.abs_path)
except cpf.DynamicLibNotFoundError as e:
    print("supported-but-absent lib ->", type(e).__name__)
print("OK — error taxonomy behaves as documented")